# Assignment 2: Advanced RAG Techniques
## Post-processing and Structured Output

In Assignment 1 you built the basic flow: load, chunk, embed, retrieve, generate. In this notebook you improve the part **between retrieval and generation**, and then make the LLM return data your code can use.

## What you will build

```
Question
   │
   ▼
Retrieve top 10          (LlamaIndex, same as before)
   │
   ▼
1. Similarity cutoff     drop chunks that are clearly off-topic
2. Metadata filter       drop chunks from the outdated policy
3. De-duplication        drop chunks that repeat the same text
4. Reranking             re-order what is left with a more accurate model
   │
   ▼
Top 3 chunks  ──►  LLM (OpenRouter)  ──►  Answer / Structured output
```

## How this notebook is organised

- **LlamaIndex** is used only for what you already know: documents, embeddings, the index and retrieval.
- **Every post-processing step is written in plain Python**, so you can see exactly what it does. Each section ends with a note on the LlamaIndex class that does the same job.
- **The LLM is called directly through OpenRouter**, as in Assignment 1.
- **The data is a small made-up company handbook** defined inside the notebook. No Google Drive is needed, and the notebook runs in under a minute.

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. Compare what you see with the **Expected result** written under each exercise
4. Each technique builds on the previous one, so complete them in order

## 0. Setup

Install the libraries, set the OpenRouter key and load the configuration.

In [1]:
!pip install -q llama-index-core llama-index-embeddings-huggingface sentence-transformers openai pydantic

In [2]:
import os

# In Google Colab: click the key icon in the left sidebar,
# add a secret named OPENROUTER_API_KEY, toggle "Notebook access" on.
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
print("✓ OpenRouter key set successfully")

✓ OpenRouter key set successfully


In [3]:
import json
import re
from datetime import date
from typing import List, Literal

# LlamaIndex: only for documents, embeddings, the index and retrieval
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.schema import NodeWithScore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Reranking model
from sentence_transformers import CrossEncoder

# LLM (OpenRouter) and structured output
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Configuration parameters for this notebook
CONFIG = {
    "llm": {
        "model": "openai/gpt-5-mini"                         # OpenRouter model used to generate answers
    },
    "embeddings": {
        "model": "BAAI/bge-small-en-v1.5"                    # Local embedding model (no API key needed)
    },
    "retrieval": {
        "top_k": 10                                          # Retrieve wide: number of chunks fetched before post-processing
    },
    "postprocessing": {
        "similarity_cutoff": 0.75,                           # Minimum similarity score a chunk needs to be kept
        "duplicate_threshold": 0.85,                         # Word overlap above which two chunks count as duplicates
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",  # Local reranking model (no API key needed)
        "final_top_n": 3                                     # Keep narrow: number of chunks sent to the LLM
    }
}

print("✓ Libraries imported and configuration loaded")

✓ Libraries imported and configuration loaded


## 1. The dummy data: a company handbook

Our data is a handbook for a made-up company called Brightwave. It has 16 short sections. Each one becomes a LlamaIndex `Document` with three pieces of metadata: `file_name`, `department` and `year`.

The data has four problems built into it on purpose. They are the same problems real document collections have:

1. **Off-topic sections.** Most sections have nothing to do with any one question.
2. **An outdated version.** The leave policy and the expense policy each exist twice: a 2023 version and a 2025 version, with different numbers.
3. **A duplicate.** The 2025 leave policy was copied almost word for word into an HR FAQ file.
4. **Sections that look relevant but don't answer.** One section repeats a common question without answering it.

Read through the data before moving on.

In [4]:
# (file_name, department, year, text)
HANDBOOK = [
    ("leave_policy_2025.md", "HR", 2025,
     "Leave Policy 2025. Full-time employees are entitled to 12 casual leave days and 10 sick leave days per calendar year. "
     "Up to 5 unused casual leave days can be carried forward to the next year. This policy is effective from 1 January 2025."),
    ("leave_policy_2023.md", "HR", 2023,
     "Leave Policy 2023. Full-time employees are entitled to 8 casual leave days and 7 sick leave days per calendar year. "
     "Unused leave cannot be carried forward to the next year. This policy is effective from 1 January 2023."),
    ("hr_faq_2025.md", "HR", 2025,
     "Leave Policy 2025. Full-time employees are entitled to 12 casual leave days and 10 sick leave days per calendar year. "
     "Up to 5 unused casual leave days can be carried forward to the next year. Effective from 1 January 2025."),
    ("leave_survey_2025.md", "HR", 2025,
     "How many days of casual leave do I get in a year? This was the most common question in the 2025 employee survey. "
     "HR has promised to explain the leave rules more clearly in the next version of the handbook."),
    ("leave_application_2025.md", "HR", 2025,
     "How to apply for casual leave: request it on the HR portal at least 2 working days in advance. Your manager approves the request. "
     "Casual leave of more than 3 days in a row also needs approval from the department head."),
    ("public_holidays_2025.md", "HR", 2025,
     "Brightwave offices are closed on 11 public holidays in 2025. The holiday calendar is published on the HR portal every December."),
    ("work_from_home_2025.md", "HR", 2025,
     "Employees can work from home up to 2 days per week. Work-from-home days must be agreed with your manager and marked in the team calendar."),
    ("notice_period_2025.md", "HR", 2025,
     "The notice period for resignation is 60 days. Unused casual leave is not paid out when an employee leaves the company."),
    ("expense_policy_2025.md", "Finance", 2025,
     "Expense Policy 2025. While travelling for work, meals are reimbursed up to Rs 1,500 per day. "
     "Expense claims must be submitted within 30 days with receipts."),
    ("expense_policy_2023.md", "Finance", 2023,
     "Expense Policy 2023. While travelling for work, meals are reimbursed up to Rs 1,000 per day. "
     "Expense claims must be submitted within 45 days with receipts."),
    ("travel_policy_2025.md", "Finance", 2025,
     "Flights for work travel must be booked through the travel desk at least 7 days in advance. "
     "Economy class is the default for flights under 6 hours."),
    ("laptop_policy_2025.md", "IT", 2025,
     "Every employee receives a company laptop on the first day. Laptops are replaced every 3 years. "
     "Lost laptops must be reported to IT within 24 hours."),
    ("password_policy_2025.md", "IT", 2025,
     "Passwords must be at least 12 characters long and must be changed every 90 days. "
     "Two-factor authentication is required for email and VPN."),
    ("cafeteria_2025.md", "Admin", 2025,
     "The cafeteria serves lunch from 12:30 pm to 2:30 pm on working days. Tea and coffee are free throughout the day."),
    ("parking_2025.md", "Admin", 2025,
     "Parking slots in the basement are given on a first-come, first-served basis. Two-wheeler parking is available near gate 2."),
    ("office_timings_2025.md", "Admin", 2025,
     "Office hours are 9:30 am to 6:30 pm, Monday to Friday. The office is closed on Saturdays and Sundays."),
]

print(f"✓ Handbook has {len(HANDBOOK)} sections")

✓ Handbook has 16 sections


### Build the index (same steps as Assignment 1)

Each handbook section becomes a `Document`, the local embedding model turns it into a vector, and `VectorStoreIndex` stores the vectors in memory. The sections are short, so each one stays as a single chunk.

In [5]:
# Embedding model (downloads the first time, then cached)
Settings.embed_model = HuggingFaceEmbedding(model_name=CONFIG["embeddings"]["model"])

# One Document per handbook section, with metadata
documents = [
    Document(text=text, metadata={"file_name": file_name, "department": department, "year": year})
    for file_name, department, year, text in HANDBOOK
]

# In-memory vector index, and a retriever that fetches the top 10 chunks
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=CONFIG["retrieval"]["top_k"])

print(f"✓ Index built with {len(documents)} documents")
print(f"✓ Retriever ready (top {CONFIG['retrieval']['top_k']})")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Index built with 16 documents
✓ Retriever ready (top 10)


In [6]:
def show_nodes(nodes, title: str = ""):
    """Print a list of retrieved chunks: rank, score, year, file name and a text preview."""
    if title:
        print(title)
    if nodes is None:
        print("  Nothing to show: the function returned None. Have you completed the TODO?")
        return
    if not nodes:
        print("  (no chunks)")
        return
    for rank, node in enumerate(nodes, 1):
        preview = node.get_content()[:70]
        print(f"  {rank:2d}. score={node.score:7.3f}  {node.metadata['year']}  {node.metadata['file_name']:<28} {preview}...")

## 2. Baseline: retrieval with no post-processing

First look at what plain retrieval returns for our main question. This is your starting point for the rest of the notebook.

In [7]:
QUESTION = "How many days of casual leave do I get in a year?"

baseline_nodes = retriever.retrieve(QUESTION)
show_nodes(baseline_nodes, f"Question: {QUESTION}\n\nTop {len(baseline_nodes)} chunks from retrieval:")

Question: How many days of casual leave do I get in a year?

Top 10 chunks from retrieval:
   1. score=  0.867  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   2. score=  0.822  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...
   3. score=  0.817  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   4. score=  0.811  2025  leave_policy_2025.md         Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   5. score=  0.797  2023  leave_policy_2023.md         Leave Policy 2023. Full-time employees are entitled to 8 casual leave ...
   6. score=  0.777  2025  notice_period_2025.md        The notice period for resignation is 60 days. Unused casual leave is n...
   7. score=  0.681  2025  work_from_home_2025.md       Employees can work from home up to 2 days per week. Work-from-home day...

### What to notice

The correct answer is **12 days**, from the 2025 leave policy. Look at the list above:

- **Rank 1 does not answer the question.** `leave_survey_2025.md` scores highest because it repeats the question almost word for word.
- **The right answer is only at rank 3 or 4**, and it appears twice (`hr_faq_2025.md` and `leave_policy_2025.md` have nearly the same text).
- **`leave_policy_2023.md` says 8 days.** It scores almost as high as the 2025 version. If it reaches the LLM, the LLM sees two different answers.
- **The last four chunks are off-topic** (work from home, travel, expenses). Notice the drop in score between rank 6 and rank 7.

If we simply sent the top 5 to the LLM, it would get one useless chunk, one duplicate, and two policies that disagree. The next four sections fix these one at a time.

## 3. Similarity cutoff

**Concept:** Retrieval always returns `top_k` chunks, even when only a few are relevant. A similarity cutoff drops every chunk whose score is below a threshold.

**Why it matters:** Off-topic chunks cost tokens and can distract the LLM. And if *nothing* passes the cutoff, you know the answer isn't in your data, so you can say so without guessing.

**How to choose the cutoff:** Look at the scores. There is no universal number, because each embedding model spreads its scores differently. With `bge-small`, even unrelated text scores around 0.5 to 0.7, so a cutoff of 0.3 would remove nothing. In the baseline above the scores drop sharply after rank 6, so we cut at 0.75.

### Your task

Complete the function below.

In [9]:
def filter_by_similarity(nodes, cutoff: float):
    """
    Keep only the chunks whose similarity score is at least `cutoff`.

    Args:
        nodes: Chunks returned by the retriever.
        cutoff: Minimum similarity score needed to keep a chunk.

    Returns:
        List of chunks that passed the cutoff, in the same order.
    """
    # TODO: Keep only the nodes whose score is at least `cutoff`
    # HINT: Use a list comprehension over `nodes` and check `node.score`
    kept = [node for node in nodes if node.score is not None and node.score >= cutoff]

    return kept

# Test the function
after_cutoff = filter_by_similarity(baseline_nodes, CONFIG["postprocessing"]["similarity_cutoff"])
show_nodes(after_cutoff, f"After similarity cutoff ({CONFIG['postprocessing']['similarity_cutoff']}):")

After similarity cutoff (0.75):
   1. score=  0.867  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   2. score=  0.822  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...
   3. score=  0.817  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   4. score=  0.811  2025  leave_policy_2025.md         Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   5. score=  0.797  2023  leave_policy_2023.md         Leave Policy 2023. Full-time employees are entitled to 8 casual leave ...
   6. score=  0.777  2025  notice_period_2025.md        The notice period for resignation is 60 days. Unused casual leave is n...


**Expected result:** 6 chunks remain. The four off-topic chunks (work from home, travel, both expense policies) are gone.

> **In LlamaIndex:** `SimilarityPostprocessor(similarity_cutoff=0.75)` does the same thing.

## 4. Metadata filtering

**Concept:** Keep only the chunks whose metadata matches a condition, for example `year == 2025`.

**Why it matters:** The 2023 and 2025 leave policies are about the same topic, so their similarity scores are almost equal. Similarity cannot tell you which one is current. Metadata can.

**Note:** Here we filter *after* retrieval so you can see the step. In a real system you usually pass the filter to the vector database, so it filters *before* searching and the outdated chunks never take up a place in the top 10.

### Your task

Complete the function below.

In [10]:
def filter_by_metadata(nodes, key: str, value):
    """
    Keep only the chunks whose metadata field `key` equals `value`.

    Args:
        nodes: List of chunks.
        key: Metadata field to check, for example "year".
        value: Value the field must have, for example 2025.

    Returns:
        List of chunks that match, in the same order.
    """
    # TODO: Keep only the nodes where the metadata field `key` equals `value`
    # HINT: `node.metadata` is a dictionary. Use `node.metadata.get(key)`
    kept = [node for node in nodes if node.metadata.get(key) == value]

    return kept

# Test the function
after_metadata = filter_by_metadata(after_cutoff, "year", 2025)
show_nodes(after_metadata, "After metadata filter (year = 2025):")

After metadata filter (year = 2025):
   1. score=  0.867  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   2. score=  0.822  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...
   3. score=  0.817  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   4. score=  0.811  2025  leave_policy_2025.md         Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   5. score=  0.777  2025  notice_period_2025.md        The notice period for resignation is 60 days. Unused casual leave is n...


**Expected result:** 5 chunks remain. `leave_policy_2023.md` is gone.

> **In LlamaIndex:** pass `filters=MetadataFilters(...)` when you create the retriever, and the vector store filters before it searches.

## 5. De-duplication

**Concept:** If two chunks say almost the same thing, keep only the first one.

**Why it matters:** Real collections are full of copies: the same paragraph in a policy and an FAQ, the same email forwarded twice, overlapping chunks. Duplicates waste the few slots you send to the LLM.

**How we measure "almost the same":** word overlap. Take the set of words in each chunk, and divide the number of shared words by the total number of different words. Identical texts give 1.0, and texts with no words in common give 0.0. This is called Jaccard similarity.

The helper `word_overlap` is given. Run the next cell to see how it behaves on our data, then complete `remove_duplicates`.

In [11]:
def word_overlap(text_a: str, text_b: str) -> float:
    """Jaccard similarity: shared words divided by all different words (0.0 to 1.0)."""
    words_a = set(re.findall(r"\w+", text_a.lower()))
    words_b = set(re.findall(r"\w+", text_b.lower()))
    return len(words_a & words_b) / len(words_a | words_b)

texts = {file_name: text for file_name, _, _, text in HANDBOOK}

print("Word overlap between pairs of handbook sections:")
for a, b in [("leave_policy_2025.md", "hr_faq_2025.md"),
             ("leave_policy_2025.md", "leave_policy_2023.md"),
             ("expense_policy_2025.md", "expense_policy_2023.md"),
             ("leave_policy_2025.md", "parking_2025.md")]:
    print(f"  {word_overlap(texts[a], texts[b]):.2f}  {a}  vs  {b}")

Word overlap between pairs of handbook sections:
  0.94  leave_policy_2025.md  vs  hr_faq_2025.md
  0.73  leave_policy_2025.md  vs  leave_policy_2023.md
  0.79  expense_policy_2025.md  vs  expense_policy_2023.md
  0.06  leave_policy_2025.md  vs  parking_2025.md


**What to notice:** the policy and its FAQ copy overlap by 0.94, so a threshold of 0.85 catches them.

Now look at the 2023 and 2025 versions of the same policy. They overlap by 0.7 to 0.8, because they differ in only a few words. But those few words are the numbers that matter. This is why we used metadata, not de-duplication, to remove the outdated policy: word overlap cannot tell which of two similar chunks is the correct one.

### Your task

Complete the function below.

In [12]:
def remove_duplicates(nodes, threshold: float):
    """
    Remove chunks that are near-duplicates of a chunk already kept.

    Go through the chunks in order. Keep a chunk only if its word overlap
    with every chunk kept so far is below `threshold`.

    Args:
        nodes: List of chunks, best first.
        threshold: Word overlap at or above which two chunks count as duplicates.

    Returns:
        List of chunks with duplicates removed, in the same order.
    """
    kept = []
    for node in nodes:
        # TODO: Check whether `node` is a duplicate of any chunk already in `kept`
        # HINT: Use any(...) with word_overlap(node.get_content(), other.get_content()) >= threshold
        is_duplicate = any(
            word_overlap(node.get_content(), other.get_content()) >= threshold
            for other in kept
        )

        if not is_duplicate:
          kept.append(node)

    return kept

# Test the function
after_dedup = remove_duplicates(after_metadata, CONFIG["postprocessing"]["duplicate_threshold"])
show_nodes(after_dedup, f"After de-duplication (threshold {CONFIG['postprocessing']['duplicate_threshold']}):")

After de-duplication (threshold 0.85):
   1. score=  0.867  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   2. score=  0.822  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...
   3. score=  0.817  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   4. score=  0.777  2025  notice_period_2025.md        The notice period for resignation is 60 days. Unused casual leave is n...


**Expected result:** 4 chunks remain. Only one of `hr_faq_2025.md` and `leave_policy_2025.md` is left (the one that ranked higher).

## 6. Reranking

**Concept:** A reranker is a second, more accurate model that reads the question and a chunk **together** and gives a relevance score. We use it to re-order the chunks that survived the filters.

**Why it matters:** The embedding model turned the question and each chunk into vectors separately, so it can only tell that two texts are about the same topic. That is why `leave_survey_2025.md` is still at rank 1: it uses the same words as the question. A reranker reads both together and can tell that the survey chunk asks the question but never answers it.

**Why not rerank everything?** The reranker must run once for every question and chunk pair, so it is far too slow for a whole collection. We let fast retrieval and the cheap filters narrow the list first, and rerank only the few chunks that are left.

**About the scores:** this reranker's scores are not on a 0 to 1 scale and can be negative. Only the order matters: higher means more relevant.

In [13]:
# Load the reranking model (downloads the first time, then cached)
reranker = CrossEncoder(CONFIG["postprocessing"]["reranker_model"])
print(f"✓ Reranker loaded: {CONFIG['postprocessing']['reranker_model']}")

# A reranker scores (question, text) pairs. Higher means more relevant.
example_scores = reranker.predict([
    (QUESTION, "Full-time employees are entitled to 12 casual leave days per calendar year."),
    (QUESTION, "The cafeteria serves lunch from 12:30 pm to 2:30 pm."),
])
print("Score for a chunk that answers the question:", round(float(example_scores[0]), 3))
print("Score for an unrelated chunk:               ", round(float(example_scores[1]), 3))

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✓ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Score for a chunk that answers the question: 8.2
Score for an unrelated chunk:                -11.307


### Your task

Complete the function below.

In [14]:
def rerank(question: str, nodes, top_n: int):
    """
    Re-order chunks by how well each one answers the question, and keep the best `top_n`.

    Args:
        question: User question.
        nodes: List of chunks to rerank.
        top_n: Number of chunks to keep after reranking.

    Returns:
        The best `top_n` chunks, most relevant first, with the reranker's score.
    """
    if not nodes:
        return []

    # TODO: Build a list of (question, chunk text) pairs, one for each node
    # HINT: Use `node.get_content()` for the chunk text
    pairs = [(question, node.get_content()) for node in nodes]

    # TODO: Score every pair with the reranker (HINT: reranker.predict)
    scores = reranker.predict(pairs)

    # Attach the new score to each chunk (given)
    reranked = [NodeWithScore(node=node.node, score=float(score)) for node, score in zip(nodes, scores)]

    # TODO: Sort `reranked` from highest score to lowest
    # HINT: reranked.sort(key=..., reverse=True)
    reranked.sort(key=lambda x: x.score, reverse=True)

    return reranked[:top_n]

# Test the function
show_nodes(after_dedup, "Before reranking (embedding similarity scores):")
final_nodes = rerank(QUESTION, after_dedup, CONFIG["postprocessing"]["final_top_n"])
show_nodes(final_nodes, f"\nAfter reranking, top {CONFIG['postprocessing']['final_top_n']} (reranker scores):")

Before reranking (embedding similarity scores):
   1. score=  0.867  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   2. score=  0.822  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...
   3. score=  0.817  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   4. score=  0.777  2025  notice_period_2025.md        The notice period for resignation is 60 days. Unused casual leave is n...

After reranking, top 3 (reranker scores):
   1. score=  7.304  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   2. score=  5.986  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   3. score=  3.424  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...

**Expected result:** the chunk with the 2025 leave numbers moves from rank 3 to rank 1. `leave_survey_2025.md` drops to rank 2, and `notice_period_2025.md` is cut.

> **In LlamaIndex:** `SentenceTransformerRerank(model=..., top_n=3)` does the same thing.

## 7. The full pipeline

Now chain the four steps into one function, and send the result to the LLM.

The order is deliberate: the cheap steps run first and the expensive one (reranking) runs last, on as few chunks as possible.

### Your task

Complete `retrieve_and_postprocess` by calling the four functions you wrote, in order.

In [15]:
def retrieve_and_postprocess(question: str, year: int = 2025, verbose: bool = True):
    """
    Retrieve chunks for a question and run all four post-processing steps.

    Args:
        question: User question.
        year: Policy year to keep in the metadata filter.
        verbose: Whether to print how many chunks are left after each step.

    Returns:
        Final list of chunks to send to the LLM.
    """
    settings = CONFIG["postprocessing"]

    nodes = retriever.retrieve(question)
    if verbose: print(f"Retrieved:               {len(nodes)} chunks")

    # TODO: Step 1 - similarity cutoff (use settings["similarity_cutoff"])
    nodes = filter_by_similarity(nodes, settings["similarity_cutoff"])
    if verbose: print(f"After similarity cutoff: {len(nodes)} chunks")

    # TODO: Step 2 - metadata filter (keep chunks where "year" equals `year`)
    nodes = filter_by_metadata(nodes, "year", year)
    if verbose: print(f"After metadata filter:   {len(nodes)} chunks")

    # TODO: Step 3 - de-duplication (use settings["duplicate_threshold"])
    nodes = remove_duplicates(nodes, settings["duplicate_threshold"])
    if verbose: print(f"After de-duplication:    {len(nodes)} chunks")

    # TODO: Step 4 - reranking (keep settings["final_top_n"] chunks)
    nodes = rerank(question, nodes, settings["final_top_n"])
    if verbose: print(f"After reranking:         {len(nodes)} chunks")

    return nodes

# Test the function
pipeline_nodes = retrieve_and_postprocess(QUESTION)
show_nodes(pipeline_nodes, "\nChunks that will be sent to the LLM:")

Retrieved:               10 chunks
After similarity cutoff: 6 chunks
After metadata filter:   5 chunks
After de-duplication:    4 chunks
After reranking:         3 chunks

Chunks that will be sent to the LLM:
   1. score=  7.304  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   2. score=  5.986  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   3. score=  3.424  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...


**Expected result:** 10 → 6 → 5 → 4 → 3 chunks, with the 2025 leave numbers at rank 1.

### Generation with OpenRouter

This is the same augmentation and generation code as Assignment 1, so it is given. One addition: if post-processing leaves no chunks, we don't call the LLM at all. We already know the handbook has nothing relevant.

In [16]:
# OpenRouter client (same openai library, different base_url)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

SYSTEM_PROMPT = (
    "You answer questions about the Brightwave company handbook. Use only the context below. "
    "Each context block starts with its source file name. "
    "If the context does not contain the answer, say that the handbook does not cover it."
)

def build_context(nodes) -> str:
    """Join chunks into one context string, each labelled with its source file."""
    return "\n\n".join(
        f"[Source: {node.metadata['file_name']}]\n{node.get_content()}" for node in nodes
    )

def generate_answer(question: str, nodes) -> str:
    """Send the chunks and the question to the LLM through OpenRouter and return the answer text."""
    if not nodes:
        return "I could not find anything about this in the handbook."

    response = client.chat.completions.create(
        model=CONFIG["llm"]["model"],
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{build_context(nodes)}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content

print(f"✓ OpenRouter client ready: {CONFIG['llm']['model']}")

✓ OpenRouter client ready: openai/gpt-5-mini


### Compare: with and without post-processing

Both calls below use the same question and the same LLM. The only difference is which chunks the LLM receives.

In [17]:
print(f"Question: {QUESTION}\n")

# Without post-processing: the top 5 chunks straight from retrieval
plain_nodes = retriever.retrieve(QUESTION)[:5]
show_nodes(plain_nodes, "WITHOUT post-processing, the LLM receives:")
print("\nAnswer:", generate_answer(QUESTION, plain_nodes))

print("\n" + "=" * 80 + "\n")

# With post-processing: the 3 chunks from the pipeline
show_nodes(pipeline_nodes, "WITH post-processing, the LLM receives:")
print("\nAnswer:", generate_answer(QUESTION, pipeline_nodes))

Question: How many days of casual leave do I get in a year?

WITHOUT post-processing, the LLM receives:
   1. score=  0.867  2025  leave_survey_2025.md         How many days of casual leave do I get in a year? This was the most co...
   2. score=  0.822  2025  leave_application_2025.md    How to apply for casual leave: request it on the HR portal at least 2 ...
   3. score=  0.817  2025  hr_faq_2025.md               Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   4. score=  0.811  2025  leave_policy_2025.md         Leave Policy 2025. Full-time employees are entitled to 12 casual leave...
   5. score=  0.797  2023  leave_policy_2023.md         Leave Policy 2023. Full-time employees are entitled to 8 casual leave ...

Answer: Under the 2025 policy, full-time employees get 12 casual leave days per calendar year (effective 1 Jan 2025).  
(For reference, the 2023 policy granted 8 casual days per year.)


WITH post-processing, the LLM receives:
   1. score=  7.30

**What to notice:** without post-processing the LLM receives both the 8-day and the 12-day policy, plus a duplicate. A strong model will often still work out the right answer, or mention both. A weaker model, or a less obvious conflict, will not. With post-processing there is only one possible answer, and the prompt is shorter.

### Try more questions

The last question below is not covered by the handbook at all. Watch what the similarity cutoff does with it.

In [18]:
def ask(question: str):
    """Run the full pipeline for a question and print the answer and its sources."""
    print(f"Question: {question}")
    nodes = retrieve_and_postprocess(question)
    print("Answer:", generate_answer(question, nodes))
    show_nodes(nodes, "Sources:")
    print("-" * 80)

ask("How much can I claim for meals when I travel for work?")
ask("How often do I need to change my password?")
ask("What is the maternity leave policy?")

Question: How much can I claim for meals when I travel for work?
Retrieved:               10 chunks
After similarity cutoff: 2 chunks
After metadata filter:   1 chunks
After de-duplication:    1 chunks
After reranking:         1 chunks
Answer: You can claim up to Rs 1,500 per day for meals while travelling for work.
Sources:
   1. score=  8.419  2025  expense_policy_2025.md       Expense Policy 2025. While travelling for work, meals are reimbursed u...
--------------------------------------------------------------------------------
Question: How often do I need to change my password?
Retrieved:               10 chunks
After similarity cutoff: 1 chunks
After metadata filter:   1 chunks
After de-duplication:    1 chunks
After reranking:         1 chunks
Answer: You must change your password every 90 days.
Sources:
   1. score=  7.158  2025  password_policy_2025.md      Passwords must be at least 12 characters long and must be changed ever...
----------------------------------------------

**Expected result:**

- **Meals:** Rs 1,500 per day. The 2023 policy (Rs 1,000) is removed by the metadata filter.
- **Password:** every 90 days, from a single chunk.
- **Maternity leave:** no chunk passes the similarity cutoff, so the pipeline returns nothing and we answer "not found" without calling the LLM. Without the cutoff, the LLM would have received ten unrelated chunks.

## 8. Structured output with Pydantic

**Concept:** So far the LLM returns free text, which is fine for a person to read. But if *code* has to use the answer (store it in a database, show it in a form, decide what to do next), free text is fragile. Structured output means the LLM replies with JSON that has exactly the fields and types we define.

**How it works here, with no framework:**

1. Define the shape you want as a **Pydantic model** (a class with typed fields).
2. Give the LLM the model's **JSON schema** and ask for JSON only.
3. **Validate** the reply with Pydantic. If a field is missing or has the wrong type, Pydantic raises an error, and we send that error back to the LLM and ask it to fix its reply.

Pydantic is the safety check: nothing reaches your code unless it matches the model.

### 8.1 A Pydantic model for a RAG answer

Our first model describes a handbook answer. Each field has a type and a description. The description is sent to the LLM as part of the schema, so write it as an instruction.

Notice the two fields a free-text answer can't give you reliably:

- `answer_found` is a `bool`, so your code can check it with a simple `if`.
- `confidence` is a `Literal`, so it can only be one of three exact values.

### Your task

Two fields are given. Add the other two.

In [19]:
class HandbookAnswer(BaseModel):
    """An answer to a question about the company handbook."""
    answer: str = Field(description="The answer to the question, in one or two sentences")
    sources: List[str] = Field(description="File names of the context blocks used for the answer. Empty list if none were used")
    # TODO: Add a field `answer_found` of type bool
    # HINT: description="True if the context contains the answer, otherwise False"
    answer_found: bool = Field(description="True if the context contains the answer, otherwise False")

    # TODO: Add a field `confidence` that can only be "low", "medium" or "high"
    # HINT: Use the type Literal["low", "medium", "high"]
    confidence: Literal["low", "medium", "high"] = Field(description="How confident the model is in its answer")

# The JSON schema is what we send to the LLM to describe the reply we want
print(json.dumps(HandbookAnswer.model_json_schema(), indent=2))

{
  "description": "An answer to a question about the company handbook.",
  "properties": {
    "answer": {
      "description": "The answer to the question, in one or two sentences",
      "title": "Answer",
      "type": "string"
    },
    "sources": {
      "description": "File names of the context blocks used for the answer. Empty list if none were used",
      "items": {
        "type": "string"
      },
      "title": "Sources",
      "type": "array"
    },
    "answer_found": {
      "description": "True if the context contains the answer, otherwise False",
      "title": "Answer Found",
      "type": "boolean"
    },
    "confidence": {
      "description": "How confident the model is in its answer",
      "enum": [
        "low",
        "medium",
        "high"
      ],
      "title": "Confidence",
      "type": "string"
    }
  },
  "required": [
    "answer",
    "sources",
    "answer_found",
    "confidence"
  ],
  "title": "HandbookAnswer",
  "type": "object"
}


**Expected result:** the printed schema lists four properties (`answer`, `sources`, `answer_found`, `confidence`), and `confidence` has an `enum` with the three allowed values.

### 8.2 Ask the LLM for JSON and validate it

`ask_structured` works for any Pydantic model. It sends the schema, asks for JSON, and validates the reply. If validation fails, it shows the LLM the error and tries again.

`response_format={"type": "json_object"}` tells the model to reply with valid JSON. That guarantees the reply is JSON, but not that it has our fields. Checking the fields is Pydantic's job.

### Your task

Complete the two TODOs in the function below.

In [21]:
def ask_structured(prompt: str, model_class, max_retries: int = 2):
    """
    Ask the LLM for a reply that matches a Pydantic model.

    Args:
        prompt: The user prompt (for RAG: the context and the question).
        model_class: Pydantic model class that describes the reply.
        max_retries: How many times to ask again if the reply fails validation.

    Returns:
        A validated instance of `model_class`.
    """
    # Get the JSON schema of the model as a string (if needed for custom system instructions,
    # though .parse handles it automatically under the hood with structured outputs)
    schema = json.dumps(model_class.model_json_schema(), indent=2)

    messages = [
        {"role": "system", "content": f"Reply with a single JSON object that strictly adheres to this schema:\n{schema}"},
        {"role": "user", "content": prompt},
    ]

    for attempt in range(1, max_retries + 2):
        try:
            # Using beta.chat.completions.parse for built-in Pydantic structured output support
            response = client.beta.chat.completions.parse(
                model=CONFIG["llm"]["model"],
                messages=messages,
                response_format=model_class,
            )
            # The parsed object is available in response.choices[0].message.parsed
            if response.choices[0].message.parsed is not None:
                return response.choices[0].message.parsed

            raw_reply = response.choices[0].message.content or ""
            return model_class.model_validate_json(raw_reply)

        except Exception as error:
            print(f"Attempt {attempt}: failed with error {error}, asking the LLM to fix it")
            # fallback message formatting
            if 'response' in locals() and response.choices[0].message.content:
                messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": f"Your output failed validation or processing:\n{error}\n"
                                                        "Reply again with corrected JSON structure only."})

    raise ValueError(f"No valid {model_class.__name__} after {max_retries + 1} attempts")


def answer_structured(question: str) -> HandbookAnswer:
    """Full RAG pipeline that returns a HandbookAnswer object, not free text."""
    nodes = retrieve_and_postprocess(question, verbose=False)
    context = build_context(nodes) or "(no relevant handbook sections were found)"
    prompt = (
        "Answer the question about the Brightwave company handbook using only the context.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    return ask_structured(prompt, HandbookAnswer)

# Test the function
result = answer_structured(QUESTION)
print(result.model_dump_json(indent=2))

{
  "answer": "Full-time employees are entitled to 12 casual leave days per calendar year.",
  "sources": [
    "hr_faq_2025.md",
    "leave_survey_2025.md"
  ],
  "answer_found": true,
  "confidence": "high"
}


**Expected result:** a JSON object with the answer (12 days), a list of source file names, `"answer_found": true` and a confidence value.

### Using the result in code

`result` is a normal Python object, not a string. Each field has the type you declared, so your program can make decisions with it.

In [22]:
for question in [QUESTION, "What is the maternity leave policy?"]:
    result = answer_structured(question)

    print(f"Question: {question}")
    if result.answer_found:                       # a real bool, no text parsing needed
        print(f"  Answer:     {result.answer}")
        print(f"  Sources:    {', '.join(result.sources)}")
        print(f"  Confidence: {result.confidence}")
    else:
        print("  Not covered by the handbook. Forwarding this question to the HR team.")
    print()

Question: How many days of casual leave do I get in a year?
  Answer:     Full-time employees receive 12 casual leave days per calendar year.
  Sources:    hr_faq_2025.md
  Confidence: high

Question: What is the maternity leave policy?
  Not covered by the handbook. Forwarding this question to the HR team.



### 8.3 Typed extraction: turning text into data

The second use of structured output is **extraction**: pulling specific values out of a document into typed fields.

Here the types do real work. `casual_leave_days` is an `int`, so you can do arithmetic with it. `effective_date` is a `date`, so you can compare it with other dates. If every field were a free-text `str`, you would gain very little over a normal answer.

The model is given. Run the cell to extract both versions of the leave policy and compare them **in Python**, with no LLM involved in the comparison.

In [23]:
class LeavePolicy(BaseModel):
    """The key numbers from one version of the leave policy."""
    policy_year: int = Field(description="The year of this policy, for example 2025")
    casual_leave_days: int = Field(description="Casual leave days per calendar year")
    sick_leave_days: int = Field(description="Sick leave days per calendar year")
    carry_forward_allowed: bool = Field(description="True if unused leave can be carried forward to the next year")
    max_carry_forward_days: int = Field(description="Maximum days that can be carried forward. 0 if carry forward is not allowed")
    effective_date: date = Field(description="Date the policy takes effect, in YYYY-MM-DD format")

def extract_leave_policy(file_name: str) -> LeavePolicy:
    """Extract a LeavePolicy object from one handbook section."""
    prompt = f"Extract the leave policy details from this text:\n\n{texts[file_name]}"
    return ask_structured(prompt, LeavePolicy)

policy_2025 = extract_leave_policy("leave_policy_2025.md")
policy_2023 = extract_leave_policy("leave_policy_2023.md")

print(policy_2025.model_dump_json(indent=2))
print(policy_2023.model_dump_json(indent=2))

# Because the fields are typed, we can calculate and compare in plain Python
extra_days = policy_2025.casual_leave_days - policy_2023.casual_leave_days
years_apart = policy_2025.effective_date.year - policy_2023.effective_date.year

print(f"\nCasual leave went up by {extra_days} days between the two policies ({years_apart} years apart).")
print(f"Total leave in 2025: {policy_2025.casual_leave_days + policy_2025.sick_leave_days} days")
if policy_2025.carry_forward_allowed and not policy_2023.carry_forward_allowed:
    print(f"Carry forward is new in 2025 (up to {policy_2025.max_carry_forward_days} days).")

{
  "policy_year": 2025,
  "casual_leave_days": 12,
  "sick_leave_days": 10,
  "carry_forward_allowed": true,
  "max_carry_forward_days": 5,
  "effective_date": "2025-01-01"
}
{
  "policy_year": 2023,
  "casual_leave_days": 8,
  "sick_leave_days": 7,
  "carry_forward_allowed": false,
  "max_carry_forward_days": 0,
  "effective_date": "2023-01-01"
}

Casual leave went up by 4 days between the two policies (2 years apart).
Total leave in 2025: 22 days
Carry forward is new in 2025 (up to 5 days).


**Expected result:** two JSON objects (12 and 10 days for 2025, 8 and 7 days for 2023), then: casual leave went up by 4 days, total leave in 2025 is 22 days, and carry forward is new in 2025.

### 8.4 What happens when the reply is wrong

This cell needs no LLM. It feeds Pydantic three replies by hand so you can see exactly what it accepts and what it rejects. This check is the reason to use Pydantic: a wrong reply is stopped here, before it reaches the rest of your program.

In [24]:
replies = {
    "A correct reply": '{"policy_year": 2025, "casual_leave_days": 12, "sick_leave_days": 10, '
                       '"carry_forward_allowed": true, "max_carry_forward_days": 5, "effective_date": "2025-01-01"}',

    "Wrong type ('twelve' is not a number)": '{"policy_year": 2025, "casual_leave_days": "twelve", "sick_leave_days": 10, '
                       '"carry_forward_allowed": true, "max_carry_forward_days": 5, "effective_date": "2025-01-01"}',

    "Missing fields": '{"policy_year": 2025, "casual_leave_days": 12}',
}

for label, reply in replies.items():
    print(f"{label}:")
    try:
        policy = LeavePolicy.model_validate_json(reply)
        print(f"  ✓ Accepted: {policy.casual_leave_days} casual leave days, effective {policy.effective_date}")
    except ValidationError as error:
        print(f"  ✗ Rejected with {error.error_count()} error(s):")
        for problem in error.errors():
            print(f"      field '{problem['loc'][0]}': {problem['msg']}")
    print()

A correct reply:
  ✓ Accepted: 12 casual leave days, effective 2025-01-01

Wrong type ('twelve' is not a number):
  ✗ Rejected with 1 error(s):
      field 'casual_leave_days': Input should be a valid integer, unable to parse string as an integer

Missing fields:
  ✗ Rejected with 4 error(s):
      field 'sick_leave_days': Field required
      field 'carry_forward_allowed': Field required
      field 'max_carry_forward_days': Field required
      field 'effective_date': Field required



**What to notice:** the first reply is accepted and turned into real Python types. The other two are rejected with a message that names the field and the problem. In `ask_structured`, this message is what we send back to the LLM so it can correct itself.

> **In LlamaIndex:** `LLMTextCompletionProgram` with a `PydanticOutputParser` wraps the same steps.
>
> **Going further:** OpenRouter also supports `response_format={"type": "json_schema", ...}`, where the model itself is forced to follow your schema. Not every model supports it, which is why this notebook uses the simpler JSON mode with Pydantic validation.

## Conclusion

You have built the post-processing stage of a RAG system by hand, and made the LLM return validated, typed data.

| Step | What it removes or fixes | What it relies on |
|---|---|---|
| Similarity cutoff | Off-topic chunks | The embedding score |
| Metadata filter | Outdated or out-of-scope chunks | Metadata you stored with each chunk |
| De-duplication | Repeated text | Word overlap between chunks |
| Reranking | Wrong order, chunks that look relevant but aren't | A second model that reads question and chunk together |
| Structured output | Free text that code can't use safely | A Pydantic model and validation |

## Experiments (try these after the session)

### Experiment 1: Move the similarity cutoff

Change `CONFIG["postprocessing"]["similarity_cutoff"]` to 0.3, then to 0.85, and run `ask(QUESTION)` each time.

- At 0.3, does the cutoff remove anything?
- At 0.85, what happens to the answer?

### Experiment 2: Remove one step

Comment out one step inside `retrieve_and_postprocess` (for example the metadata filter) and run the comparison again.

- Which chunks reach the LLM now?
- Does the answer change?

### Experiment 3: Filter by a different field

Use `filter_by_metadata(nodes, "department", "IT")` on the results of a broad question such as "What rules do I need to follow?".

- When would filtering by department be useful in a real system?

### Experiment 4: Your own Pydantic model

Write a model called `ExpensePolicy` with typed fields for the daily meal limit, the number of days allowed to submit a claim, and the policy year. Extract it from both expense policy sections with `ask_structured`, and calculate how much the meal limit increased.

In [ ]:
# Experiment area

experiment_question = "Replace this with your own question"

# ask(experiment_question)

## Reflection questions

Answer these in a markdown cell below after you complete the notebook.

1. Why is a similarity cutoff of 0.3 useless with this embedding model? How would you choose a cutoff for a new model?
2. The 2023 and 2025 leave policies have almost the same similarity score. Why can't the embedding model tell which one is current?
3. Why did we remove the outdated policy with metadata and not with de-duplication?
4. Why does the reranker rank the survey chunk lower than the embedding model did?
5. Why do we rerank only the few chunks left after filtering, and not the whole handbook?
6. What does the pipeline do when no chunk passes the similarity cutoff? Why is that better than sending chunks to the LLM anyway?
7. `response_format={"type": "json_object"}` already guarantees valid JSON. What does Pydantic add on top of that?
8. Give one example from your own work where a typed field (a number, a date, a yes/no) would be more useful than a free-text answer.

In [ ]:
# Write short answers here as comments or create a markdown cell below.

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.